# Experimento D: Interpretabilidad y Caja Blanca (XAI)

## Objetivo
Cumplir el Objetivo Específico 6: **"Analizar la importancia de las características"**.  
No basta con decir "es fraude", hay que decir **por qué**.

## Técnicas

| Técnica | Método | Entregable |
|---------|--------|------------|
| **Feature Importance Nativa** | Gain, weight, cover de XGBoost | Gráfico Top-10 + comparación tipos |
| **SHAP Values (global)** | TreeExplainer sobre muestra de test | Beeswarm + tabla mean \|SHAP\| |
| **SHAP Values (local)** | Force plots con contexto de transacción | 2 Force plots documentados |
| **Dependence plots** | Interacciones entre variables | Dependence TX_AMOUNT / TERMINAL_ID_RISK |

## Modelo Utilizado
**XGBoost baseline** (mejor AUPRC del Experimento A) en lugar de cost-sensitive, para interpretar el modelo de referencia con mayor rendimiento.

## Métricas del Modelo
- AUPRC, AUC ROC, Card Precision@100

In [ ]:
import os
import sys
import warnings
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn import metrics
import xgboost as xgb
import shap

# Configuración del proyecto
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from experiments.config import (
    SEED, INPUT_FEATURES, OUTPUT_FEATURE,
    BASELINE_PARAMS, RESULTS_DIR, FIGURES_DIR, COLORS, TOP_K_LIST,
    START_DATE_TRAINING, DELTA_TRAIN, DELTA_DELAY, DELTA_TEST,
)
from experiments.data_utils import (
    load_transformed_data, get_train_test_set,
    print_dataset_summary, card_precision_top_k,
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid', {'axes.facecolor': '0.9'})

print("=" * 60)
print("  EXPERIMENTO D: INTERPRETABILIDAD Y XAI")
print("=" * 60)
print(f"  Semilla: {SEED}")
print(f"  Modelo: XGBoost baseline (mejor AUPRC)")
print(f"  Técnicas: Feature Importance (Gain/weight/cover) + SHAP + Dependence")

---
## 1. Preparación de Datos y Modelo

In [ ]:
transactions_df = load_transformed_data()
train_df, test_df = get_train_test_set(
    transactions_df,
    start_date_training=START_DATE_TRAINING,
    delta_train=DELTA_TRAIN,
    delta_delay=DELTA_DELAY,
    delta_test=DELTA_TEST,
)
print_dataset_summary(train_df, test_df, "Experimento D - Interpretabilidad")

# Mejora: usar XGBoost BASELINE (mejor AUPRC que cost-sensitive)
model_xgb = xgb.XGBClassifier(**BASELINE_PARAMS["XGBoost"])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_df[INPUT_FEATURES])
X_test_scaled = scaler.transform(test_df[INPUT_FEATURES])

model_xgb.fit(X_train_scaled, train_df[OUTPUT_FEATURE])

# Métricas del modelo
y_pred_proba = model_xgb.predict_proba(X_test_scaled)[:, 1]
auprc = metrics.average_precision_score(test_df[OUTPUT_FEATURE], y_pred_proba)
auc_roc = metrics.roc_auc_score(test_df[OUTPUT_FEATURE], y_pred_proba)

predictions_d_df = test_df.copy()
predictions_d_df['predictions'] = y_pred_proba
_, _, cp100 = card_precision_top_k(predictions_d_df, top_k=100)

print(f"\n  XGBoost baseline entrenado:")
print(f"    AUC ROC:  {auc_roc:.4f}")
print(f"    AUPRC:    {auprc:.4f}")
print(f"    CP@100:   {cp100:.4f}")

---
## 2. Feature Importance Nativa (Gain, weight, cover)

In [ ]:
# Extraer feature importances: Gain (default), weight, cover
importances = model_xgb.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': INPUT_FEATURES,
    'Gain': importances,
}).sort_values('Gain', ascending=False)

booster = model_xgb.get_booster()
for imp_type in ['weight', 'cover']:
    scores = booster.get_score(importance_type=imp_type)
    feat_map = {f'f{i}': name for i, name in enumerate(INPUT_FEATURES)}
    feature_importance_df[imp_type] = feature_importance_df['Feature'].map(
        {feat_map.get(k, k): v for k, v in scores.items()}
    ).fillna(0)

# Top 10 variables (Gain)
top_10 = feature_importance_df.head(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(top_10)), top_10['Gain'].values, color=COLORS['baseline'])
ax.set_yticks(range(len(top_10)))
ax.set_yticklabels(top_10['Feature'].values, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('Importancia (Gain)', fontsize=12)
ax.set_title('Top-10 Variables Más Importantes\n(XGBoost Baseline)', fontsize=14)
for i, val in enumerate(top_10['Gain']):
    ax.text(val + 0.002, i, f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_d_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop-10 (Gain), comparación weight/cover:")
display(feature_importance_df.head(10).reset_index(drop=True))

---
## 3. SHAP Values - Análisis Global

In [ ]:
# Calcular SHAP values sobre una muestra ampliada (1000) para mayor estabilidad
SAMPLE_SIZE = min(1000, len(X_test_scaled))
np.random.seed(SEED)
sample_indices = np.random.choice(len(X_test_scaled), SAMPLE_SIZE, replace=False)
X_sample = X_test_scaled[sample_indices]

explainer = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X_sample)

# DataFrame con nombres de features
X_sample_df = pd.DataFrame(X_sample, columns=INPUT_FEATURES)

# Tabla mean |SHAP| por variable (análisis cuantitativo)
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
shap_impact_df = pd.DataFrame({
    'Feature': INPUT_FEATURES,
    'mean_abs_SHAP': mean_abs_shap,
}).sort_values('mean_abs_SHAP', ascending=False)
shap_impact_df.to_csv(RESULTS_DIR / 'experiment_d_shap_mean_impact.csv', index=False)

print(f"SHAP calculado para {SAMPLE_SIZE} muestras. Top variables por mean |SHAP|:")
display(shap_impact_df.head(10))

In [ ]:
# Beeswarm plot (resumen global de importancia SHAP)
fig = plt.figure(figsize=(12, 7))
shap.summary_plot(shap_values, X_sample_df, show=False)
plt.title(f'SHAP Beeswarm - XGBoost Baseline ({SAMPLE_SIZE} muestras)\n', fontsize=14)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_d_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. SHAP Force Plots - Explicaciones Individuales

In [ ]:
# Encontrar ejemplos y documentar contexto (monto, terminal, etc.)
y_test_sample = test_df[OUTPUT_FEATURE].iloc[sample_indices].values
preds_sample = model_xgb.predict_proba(X_sample)[:, 1]
test_sample_df = test_df.iloc[sample_indices].reset_index(drop=True)

def describe_transaction(idx, label):
    row = test_sample_df.iloc[idx]
    ctx = []
    if 'TX_AMOUNT' in row: ctx.append(f"TX_AMOUNT={row['TX_AMOUNT']:.2f}")
    if 'TERMINAL_ID' in row: ctx.append(f"TERMINAL_ID={row['TERMINAL_ID']}")
    if 'CUSTOMER_ID' in row: ctx.append(f"CUSTOMER_ID={row['CUSTOMER_ID']}")
    for f in ['TERMINAL_ID_RISK_7DAY_WINDOW', 'TERMINAL_ID_RISK_1DAY_WINDOW']:
        if f in row: ctx.append(f"{f}={row[f]:.4f}")
    return f"{label}: " + ", ".join(ctx)

# Transacción FRAUDULENTA con mayor probabilidad
fraud_indices = np.where(y_test_sample == 1)[0]
if len(fraud_indices) > 0:
    fraud_idx = fraud_indices[np.argmax(preds_sample[fraud_indices])]
    fraud_ctx = describe_transaction(fraud_idx, "FRAUDE")
    print(f"Ejemplo FRAUDE (índice {fraud_idx}): {fraud_ctx}")
    print(f"  Probabilidad predicción: {preds_sample[fraud_idx]:.4f}")
    fig = plt.figure(figsize=(16, 3))
    shap.force_plot(
        explainer.expected_value, shap_values[fraud_idx],
        X_sample_df.iloc[fraud_idx], matplotlib=True, show=False
    )
    plt.title(f'Force Plot - Transacción FRAUDULENTA\n({fraud_ctx})', fontsize=11)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'experiment_d_shap_force_fraud.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠ No se encontraron fraudes en la muestra")
    fraud_ctx = None

# Transacción NORMAL con menor probabilidad de fraude
normal_indices = np.where(y_test_sample == 0)[0]
normal_idx = normal_indices[np.argmin(preds_sample[normal_indices])]
normal_ctx = describe_transaction(normal_idx, "NORMAL")
print(f"\nEjemplo NORMAL (índice {normal_idx}): {normal_ctx}")
print(f"  Probabilidad predicción: {preds_sample[normal_idx]:.4f}")

fig = plt.figure(figsize=(16, 3))
shap.force_plot(
    explainer.expected_value, shap_values[normal_idx],
    X_sample_df.iloc[normal_idx], matplotlib=True, show=False
)
plt.title(f'Force Plot - Transacción NORMAL\n({normal_ctx})', fontsize=11)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_d_shap_force_normal.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Dependence Plots (interacciones entre variables)

In [ ]:
# Dependence plots para analizar interacciones (p. ej. TX_AMOUNT vs TERMINAL_ID_RISK)
for dep_feat in ['TX_AMOUNT', 'TERMINAL_ID_RISK_7DAY_WINDOW']:
    if dep_feat in INPUT_FEATURES:
        try:
            fig = plt.figure(figsize=(10, 5))
            shap.dependence_plot(dep_feat, shap_values, X_sample_df, show=False)
            plt.title(f'SHAP Dependence: {dep_feat}', fontsize=12)
            plt.tight_layout()
            safe_name = dep_feat.lower().replace(' ', '_')
            fig.savefig(FIGURES_DIR / f'experiment_d_shap_dependence_{safe_name}.png', dpi=150, bbox_inches='tight')
            plt.show()
            print(f"Dependence plot {dep_feat} guardado")
        except Exception as e:
            print(f"Dependence plot {dep_feat}: {e}")

---
## 6. Guardar Resultados

In [ ]:
# Guardar resultados
feature_importance_df.to_csv(RESULTS_DIR / 'experiment_d_feature_importance.csv', index=False)
if 'weight' in feature_importance_df.columns and 'cover' in feature_importance_df.columns:
    feature_importance_df.to_csv(RESULTS_DIR / 'experiment_d_feature_importance_all_types.csv', index=False)

results_d = {
    'feature_importance': feature_importance_df,
    'shap_mean_impact': shap_impact_df,
    'shap_values': shap_values,
    'X_sample': X_sample_df,
    'metrics': {'auc_roc': auc_roc, 'auprc': auprc, 'card_precision_at_100': cp100},
    'metadata': {
        'model': 'XGBoost baseline',
        'seed': SEED,
        'shap_sample_size': SAMPLE_SIZE,
        'force_plot_fraud_context': fraud_ctx,
        'force_plot_normal_context': normal_ctx,
    },
}
with open(RESULTS_DIR / 'experiment_d_results.pkl', 'wb') as f:
    pickle.dump(results_d, f)

print("✓ Resultados del Experimento D guardados exitosamente")
print(f"  - Feature importance: {RESULTS_DIR / 'experiment_d_feature_importance.csv'}")
print(f"  - SHAP mean |impact|: {RESULTS_DIR / 'experiment_d_shap_mean_impact.csv'}")
print(f"  - Resultados PKL: {RESULTS_DIR / 'experiment_d_results.pkl'}")
print(f"  - Figuras: {FIGURES_DIR}")

---
## 7. Conclusiones del Experimento D

**Entregables generados (mejorado según crítica):**
1. Gráfico Top-10 variables con Feature Importance Gain, weight y cover
2. Tabla mean |SHAP| por variable (análisis cuantitativo)
3. Beeswarm plot SHAP (1000 muestras)
4. Force plots con contexto de transacción (monto, terminal, cliente)
5. Dependence plots para interacciones (TX_AMOUNT, TERMINAL_ID_RISK_7DAY_WINDOW)

Modelo: **XGBoost baseline** (mejor AUPRC que cost-sensitive). Estos entregables demuestran **qué** variables son importantes y **en qué dirección** influyen en la predicción de fraude.